## objectives
- Load one of the checkpoint files
- Reconfigure some hyperparameters
- Copy tensorboard logs from the previous run
- Resume training with a copy of the previous tensorboard logs

In [1]:
from collections import Counter

import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode



In [ ]:
data_dir = '/home/takayuki/Desktop/summer2025/plants/data/AquaticPlantLabData/squared'

train_percent = 0.8
batch_size = 1

random_seed = 8

In [ ]:
torch.manual_seed(random_seed)

# basic_tf = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    
# ])

aug_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.5, 1.0), ratio=(1.0, 1.0)),
    
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.01),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.5, interpolation=InterpolationMode.BICUBIC),    
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


dataset = datasets.ImageFolder(data_dir, transform = aug_tf)

train_size = int(train_percent * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size], 
                                                              generator=torch.Generator().manual_seed(random_seed))
train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size, shuffle=False)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}\n")

# print how many images and of which classes are in the train and test datasets

def count_classes(subset):
    # Get the labels for the indices in the subset
    labels = [subset.dataset.targets[i] for i in subset.indices]
    label_counts = Counter(labels)
    return label_counts

train_class_counts = count_classes(train_dataset)
test_class_counts = count_classes(test_dataset)
class_names = dataset.classes
max_len = max(len(name) for name in class_names)
for i, class_name in enumerate(class_names):
    train_count = train_class_counts.get(i, 0)
    test_count = test_class_counts.get(i, 0)
    class_name = class_name.ljust(max_len+1)
    print(f"{class_name}: Train: {train_count}\t, Test: {test_count}")


## Load pretrained model
- ResNet50 with ImageNet1K weights

In [ ]:
from tqdm import tqdm
from datetime import datetime

import timm
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary

In [ ]:
# list resnet models available in timm
timm.list_models('*efficientnetv2_b0*', pretrained=True)

In [ ]:
model = timm.create_model('tf_efficientnetv2_b0.in1k', pretrained=True)
summary(model, input_size=(1, 3, 224, 224))

In [ ]:
print(model)

In [ ]:
model.classifier = nn.Linear(model.classifier.in_features, len(dataset.classes))

for param in model.parameters():
    param.requires_grad = False
    
for param in model.classifier.parameters():
    param.requires_grad = True
    
print("Number of trainable parameters: ", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
checkpoint_path = "/home/takayuki/Desktop/summer2025/plants/training/checkpoints"
models_path = "/home/takayuki/Desktop/summer2025/plants/training/models"
base_logs_path = "/home/takayuki/Desktop/summer2025/plants/training/runs/efficientnetv2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
print(f"Using device: {device}, {torch.cuda.get_device_name(device) if device.type == 'cuda' else 'CPU'}")

## Resume Training from a previous checkpoint

In [ ]:
# Find desired checkpoint
from tabnanny import check

from scipy import optimize


checkpoint_name = "checkpoint_06-18_12-50--03_more_ep_aug_efficientnetv2_EfficientNet_epoch_49"

checkpoint_file_path = os.path.join(checkpoint_path, checkpoint_name + ".pth")
if not os.path.exists(checkpoint_file_path):
    raise FileNotFoundError(f"Checkpoint file {checkpoint_file_path} does not exist.")

print(f"Loading checkpoint from {checkpoint_file_path}")
checkpoint = torch.load(checkpoint_file_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

## Training Loop
- Tensorboard logging
- Save model checkpoints
- Save best models

#### the following block is used when we are trainig from 0 epochs (not resuming from a checkpoint)

In [ ]:
# # Parameters
# run_name = "aug_efficientnetv2"
# learning_rate = 0.001
# num_epochs = 30

# checkpoint_interval = 7
# validation_interval = 1

# # Paths
# checkpoint_path = "/home/takayuki/Desktop/summer2025/plants/training/checkpoints"
# models_path = "/home/takayuki/Desktop/summer2025/plants/training/models"
# base_logs_path = "/home/takayuki/Desktop/summer2025/plants/training/runs/efficientnetv2"

# os.makedirs(checkpoint_path, exist_ok=True)
# os.makedirs(models_path, exist_ok=True)
# os.makedirs(base_logs_path, exist_ok=True)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model.to(device)
# print(f"Using device: {device}, {torch.cuda.get_device_name(device) if device.type == 'cuda' else 'CPU'}")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=learning_rate)

now = datetime.now()
timestamp = now.strftime("%m-%d_%H-%M--%S")

model_name = model.name if hasattr(model, 'name') else model.__class__.__name__

hparams_dict = {
    "a_run_name": run_name,
    "learning_rate": learning_rate,
    "num_epochs": num_epochs,
    "batch_size": batch_size,
    "train_dataset_size": len(train_dataset),
    "test_dataset_size": len(test_dataset),
    "model_architecture": model_name,
    "optimizer": optimizer.__class__.__name__,
    "criterion": criterion.__class__.__name__,
    "timestamp": timestamp,
}

writer = SummaryWriter(log_dir=os.path.join(base_logs_path, f"{timestamp}_{run_name}_{model_name}"))

for key, value in hparams_dict.items():
    writer.add_text(key, str(value))

# Training loop
best_val_accuracy = 0.0
best_val_epoch = 0

print("Starting training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{num_epochs}", unit="batch"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / total
    train_accuracy = correct / total
    
    print(f"Training Loss: {train_loss:.4f}, Training Accuracy: {train_accuracy*100:.4f}%")
    
    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_accuracy, epoch)
    
    # Validation phase
    if (epoch + 1) % validation_interval == 0:
        model.eval()  
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():  
            
            for val_images, val_labels in test_dataloader:
                val_images, val_labels = val_images.to(device), val_labels.to(device)
                
                val_outputs = model(val_images)
                val_loss = criterion(val_outputs, val_labels)
                
                val_running_loss += val_loss.item() * val_images.size(0)
                _, val_predicted = torch.max(val_outputs.data, 1)
                val_total += val_labels.size(0)
                val_correct += (val_predicted == val_labels).sum().item()
        
        val_loss = val_running_loss / val_total
        val_accuracy = val_correct / val_total

        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy*100:.4f}%")
        
        writer.add_scalar('Loss/validation', val_loss, epoch)
        writer.add_scalar('Accuracy/validation', val_accuracy, epoch)
        
        # best model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_val_epoch = epoch + 1
            
            # Delete previous best model if exists
            best_model_name = f"best_model_{model_name}_{timestamp}.pth"
            best_model_file_path = os.path.join(models_path, best_model_name)
            
            if os.path.exists(best_model_file_path):
                os.remove(best_model_file_path)
            
            # Save the new one
            torch.save(model.state_dict(), best_model_file_path)
            print(f"Saved new best model: {best_model_file_path}")
        
        
    # Checkpoint saving phase
    if (epoch + 1) % checkpoint_interval == 0:
        checkpoint_name = f"checkpoint_epoch_{epoch + 1}_{model_name}_{timestamp}.pth"
        checkpoint_file_path = os.path.join(checkpoint_path, checkpoint_name)
        
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss, 
        }, checkpoint_file_path)
        
        print(f"Saved checkpoint: {checkpoint_file_path}")

print("\nTraining complete.")
  

In [ ]:
observations = f"Training looks promising, more epochs, could help"
hparams_dict["observations"] = observations
writer.add_text('Observations', observations)

# Save the final model
final_model_name = f"final_model_{model_name}_{timestamp}.pth"
final_model_file_path = os.path.join(models_path, final_model_name)
torch.save(model.state_dict(), final_model_file_path)
print(f"Saved final model: {final_model_file_path}")

# Write Metrics and Hyperparameters to TensorBoard
metrics = {
    "epochs_trained": epoch + 1,
    "final_train_loss": train_loss,
    "final_train_accuracy": train_accuracy,
    "final_val_loss": val_loss,
    "final_val_accuracy": val_accuracy,
    "best_val_accuracy": best_val_accuracy,
    "best_val_epoch": best_val_epoch,
}
writer.add_hparams(hparams_dict, metrics)
writer.flush()
writer.close()  